# 01. Первичная разведка набора USDA Branded Food
Цель: убедиться в окружении, оценить объём, типы, пропуски и уникальность, составить перечень дефектов (план очистки). Работа идёт на отладочном срезе (200 тыс. строк), полный набор обрабатывается скриптом `src.pipeline`.

## 1. Окружение
Проверяем, что ядро блокнота запущено интерпретатором Python 3.12.14 из `.venv`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import platform
print(platform.python_version())
print(sys.executable)
print('виртуальное окружение активно:', sys.prefix != sys.base_prefix)

## 2. Файл и срез
Срез формируется потоковым чтением (в память попадает одна строка за раз).

In [ ]:
from src import config as cfg, io_utils as io
if not cfg.DEBUG_SLICE.exists():
    io.make_slice()
info = io.file_info(cfg.RAW_BRANDED)
print({k: v for k, v in info.items() if k != 'columns'})
info['columns']

## 3. Загрузка среза «как есть» (типы подбирает pandas)

In [ ]:
import pandas as pd
from IPython.display import display
pd.set_option('display.max_columns', 50); pd.set_option('display.width', 200)
df = io.read_csv_naive(cfg.DEBUG_SLICE)
print(df.shape)
display(df.head(5))

## 4. Типы, память, пропуски, уникальность
Доля уникальных значений — критерий перевода в `category`: меньше 50 % — категория; почти уникальные столбцы (идентификаторы, длинный текст) в категорию не переводятся.

In [ ]:
prof = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'unique': df.nunique(),
    'unique_share': (df.nunique() / len(df)).round(4),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'memory_mb': (df.memory_usage(deep=True, index=False) / 1e6).round(2)})
display(prof)
print('Всего в памяти, МБ:', round(df.memory_usage(deep=True).sum() / 1e6, 1))

## 5. Скрытые дефекты
Заглушки вместо пропусков, служебные символы, регистр, ведущие нули GTIN.

In [ ]:
placeholders = {'n/a', 'na', 'none', 'null', 'nan', '-', '--', 'unknown'}
rows = []
for c in df.select_dtypes(exclude='number').columns:
    s = df[c].dropna().astype(str)
    rows.append(dict(column=c,
                     placeholders=int(s.str.strip().str.lower().isin(placeholders).sum()),
                     edge_spaces=int((s != s.str.strip()).sum()),
                     double_spaces=int(s.str.contains('  ').sum()),
                     html_like=int(s.str.contains('<[^>]+>', regex=True).sum())))
display(pd.DataFrame(rows).set_index('column'))
print('gtin_upc прочитан как', df['gtin_upc'].dtype, '- ведущие нули утеряны:', df['gtin_upc'].head(3).tolist())
display(df['serving_size_unit'].value_counts())
display(df['serving_size'].describe())

## 6. Перечень дефектов (план очистки)
См. `DATASET.md`, раздел «Дефекты». Кратко: неверный тип `gtin_upc`; заглушки `n/a`, `none`, `nan`; пробелы и регистр в `brand_owner`; разные коды единиц (`g/GRM/GM`, `ml/MLT`); JSON-список в `trade_channel`; HTML в `ingredients`; нулевые и огромные значения `serving_size`; несколько версий одного GTIN.